# BA283 A/B — FiLM vs plain GRU (concat)
Reads the expanded dataset straight from Google Drive (staged by the box).
Trains both conditionings on identical data, compares val ESR, exports both.


## 1. Mount Drive + locate dataset


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import glob, os
cands = glob.glob('/content/drive/**/ba283_spice', recursive=True)
assert cands, 'ba283_spice not found under Drive'
DATA = cands[0]; print('DATA =', DATA)
assert os.path.exists(os.path.join(DATA,'manifest.csv')), 'manifest.csv missing (batch not done?)'
print('rows:', sum(1 for _ in open(os.path.join(DATA,'train.csv')))-1, 'train')


## 2. Get the training code + deps
Paste a GitHub token with read access (kept out of output via getpass).


In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ')
import subprocess
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch; print('cuda:', torch.cuda.is_available())


## 3. Rewrite manifest paths to the Colab dataset location
The box wrote absolute paths; remap each to DATA/<wet|dry>/<basename>.


In [ ]:
import pandas as pd, os
for name in ['train','val','test']:
    df = pd.read_csv(os.path.join(DATA, name+'.csv'))
    for col in ['dry_path','wet_path']:
        df[col] = df[col].apply(lambda x: os.path.join(DATA, os.path.basename(os.path.dirname(x)), os.path.basename(x)))
    df.to_csv('/content/%s.csv'%name, index=False)
print(pd.read_csv('/content/train.csv').iloc[0].to_dict())
assert os.path.exists(pd.read_csv('/content/train.csv').iloc[0]['wet_path']), 'path remap failed'


## 4. Train FiLM


In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning film \
  --checkpoint-dir ckpt_film --log-dir logs_film --patience 40 --seed 42


## 5. Train plain GRU (concat)


In [ ]:
!python -m train.train --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 1e-3 --batch 128 --stride 4096 --hidden 32 --device cuda --conditioning concat \
  --checkpoint-dir ckpt_concat --log-dir logs_concat --patience 40 --seed 42


## 6. Compare val ESR + export both to Drive


In [ ]:
import torch, numpy as np
from train.dataset import make_loader
from train.model import BA283GRU
from train.export import export_rtneural_json
from train.loss import esr
def evalm(cond, ckpt):
    m = BA283GRU(32, conditioning=cond)
    m.load_state_dict(torch.load(ckpt+'/best.pth', map_location='cpu', weights_only=True)); m.eval()
    ld = make_loader('/content/val.csv', batch_size=32, window=4096, stride=2048, num_workers=2, shuffle=False, drop_last=False)
    P=[];T=[];warm=1024
    with torch.no_grad():
        for x,y in ld:
            _,h=m(x[:,:warm]); p,_=m(x[:,warm:],h); P.append(p.reshape(-1)); T.append(y[:,warm:].reshape(-1))
    e=esr(torch.cat(P),torch.cat(T)).item()
    export_rtneural_json(m, DATA+'/ba283_%s.json'%cond, meta={'source':'colab-ab','conditioning':cond})
    return e
res = {c: evalm(c, ck) for c,ck in [('film','ckpt_film'),('concat','ckpt_concat')]}
for c,e in res.items(): print('%-7s val ESR = %.4f (%.1f dB)' % (c, e, 10*np.log10(max(e,1e-12))))
win = min(res, key=res.get); print('WINNER:', win)
print('exported ba283_film.json + ba283_concat.json to', DATA)


Bring back `ba283_<winner>.json` (on Drive). I embed it + port that conditioning to the Rust plugin (#30).
